In [1]:
# 1. Импорт необходимых библиотек
# Если yfinance не установлен, выполните в терминале: pip install yfinance pandas
import yfinance as yf
import pandas as pd

# 2. Загрузка данных по тикеру CME Bitcoin Futures
# period="max" загрузит все доступные исторические данные (с декабря 2017 года)
# interval="1d" гарантирует ежедневную частоту (daily)
print("Загрузка данных с Yahoo Finance...")
df_futures = yf.download("BTC=F", period="max", interval="1d", progress=False)

# 3. Проверка и обработка структуры данных
# yfinance иногда возвращает многоуровневые колонки (MultiIndex), 
# поэтому мы сбрасываем их до одноуровневых для простоты работы
if isinstance(df_futures.columns, pd.MultiIndex):
    df_futures.columns = df_futures.columns.get_level_values(0)

# Сбрасываем индекс, чтобы дата стала обычным столбцом с именем 'Date'
df_futures = df_futures.reset_index()

# 4. Переименование столбцов в соответствии с требуемым форматом
# Используем префикс "F_" (Futures), чтобы отличать их от спотовых данных при будущем объединении
# Примечание: используем "F_Open_Interest" вместо "F_Open Interest", так как пробелы в именах 
# столбцов pandas усложняют обращение к ним (требуют df['F_Open Interest'] вместо df.F_Open_Interest)
column_mapping = {
    'Date': 'Date',
    'Open': 'F_Open',
    'High': 'F_High',
    'Low': 'F_Low',
    'Close': 'F_Close',
    'Volume': 'F_Volume',
    'Open Interest': 'F_Open_Interest'
}

# Переименовываем только те колонки, которые реально присутствуют в загруженных данных
existing_cols = {k: v for k, v in column_mapping.items() if k in df_futures.columns}
df_futures = df_futures.rename(columns=existing_cols)

# 5. Первичный осмотр результата
print(f"\nРазмер датасета: {df_futures.shape[0]} строк, {df_futures.shape[1]} столбцов")
print("Первые 5 строк данных:")
display(df_futures.head())

print("\nИнформация о типах данных и пропусках:")
df_futures.info()

Загрузка данных с Yahoo Finance...

Размер датасета: 2163 строк, 6 столбцов
Первые 5 строк данных:


Price,Date,F_Close,F_High,F_Low,F_Open,F_Volume
0,2017-12-18,19100.0,20650.0,18345.0,20650.0,1054
1,2017-12-19,18200.0,19725.0,17180.0,19135.0,559
2,2017-12-20,17040.0,18350.0,16435.0,17745.0,784
3,2017-12-21,15330.0,17270.0,15080.0,16400.0,879
4,2017-12-22,14135.0,15825.0,12265.0,15595.0,2374



Информация о типах данных и пропусках:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2163 entries, 0 to 2162
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      2163 non-null   datetime64[ns]
 1   F_Close   2163 non-null   float64       
 2   F_High    2163 non-null   float64       
 3   F_Low     2163 non-null   float64       
 4   F_Open    2163 non-null   float64       
 5   F_Volume  2163 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 101.5 KB


In [2]:
# Задаем имя файла (можно изменить на любое удобное)
file_name = "cme_btc_futures_daily.csv"

# Сохраняем DataFrame в CSV
# index=False非常重要: это предотвращает сохранение номеров строк (индекса pandas) 
# как отдельного лишнего столбца в файле
df_futures.to_csv(file_name, index=False, encoding='utf-8')

print(f"Данные успешно сохранены в файл: {file_name}")

Данные успешно сохранены в файл: cme_btc_futures_daily.csv


In [3]:
# 1. Переименуем исходный столбец, чтобы зафиксировать, что это количество контрактов
df_futures = df_futures.rename(columns={'F_Volume': 'F_Volume_Contracts'})

# 2. Рассчитаем реальный долларовый объем торгов
# Умножаем контракты на цену закрытия и на 5 (стандартный размер контракта CME)
df_futures['F_Volume_USD'] = df_futures['F_Volume_Contracts'] * df_futures['F_Close'] * 5

# 3. Для удобства восприятия создадим столбец в миллионах долларов (опционально, но рекомендуется для графиков)
df_futures['F_Volume_USD_Mln'] = df_futures['F_Volume_USD'] / 1_000_000

# 4. Проверим результат на последних строках
print("Проверка корректности расчета:")
cols_to_check = ['Date', 'F_Close', 'F_Volume_Contracts', 'F_Volume_USD', 'F_Volume_USD_Mln']
display(df_futures[cols_to_check].tail())

Проверка корректности расчета:


Price,Date,F_Close,F_Volume_Contracts,F_Volume_USD,F_Volume_USD_Mln
2158,2026-07-20,65185.0,8793,2.865859e+09,2865.858525
2159,2026-07-21,66520.0,8577,2.852710e+09,2852.710200
2160,2026-07-22,66010.0,5126,1.691836e+09,1691.836300
2161,2026-07-23,64855.0,5126,1.662234e+09,1662.233650
2162,2026-07-24,65695.0,1209,3.971263e+08,397.126275


In [4]:
# Задаем имя файла (можно изменить на любое удобное)
file_name = "cme_btc_futures_daily.csv"

# Сохраняем DataFrame в CSV
# index=False非常重要: это предотвращает сохранение номеров строк (индекса pandas) 
# как отдельного лишнего столбца в файле
df_futures.to_csv(file_name, index=False, encoding='utf-8')

print(f"Данные успешно сохранены в файл: {file_name}")

Данные успешно сохранены в файл: cme_btc_futures_daily.csv


In [1]:
import pandas as pd

# 1. Загружаем текущую версию датасета фьючерсов CME
df_cme = pd.read_csv("cme_btc_futures_daily.csv")

# 2. Удаляем столбец F_Volume_USD, если он существует
# Это избавляет от избыточных огромных чисел, оставляя более удобные метрики
if 'F_Volume_USD' in df_cme.columns:
    df_cme = df_cme.drop(columns=['F_Volume_USD'])
    print("✅ Столбец 'F_Volume_USD' успешно удален.")

# 3. Округляем все числовые столбцы до 1 знака после запятой
# select_dtypes автоматически найдет все колонки с типами float или int
numeric_cols = df_cme.select_dtypes(include=['float64', 'int64']).columns.tolist()

for col in numeric_cols:
    df_cme[col] = df_cme[col].round(1)

# 4. Сохраняем финальную версию датасета фьючерсов
final_cme_file = "cme_btc_futures_daily_final.csv"
df_cme.to_csv(final_cme_file, index=False, encoding='utf-8')

print(f"✅ Датасет фьючерсов успешно обновлен и сохранен как: {final_cme_file}")
print("\nПервые 5 строк обновленного датасета (обратите внимание на округление):")
display(df_cme.head())

print("\nИнформация о типах данных и размере:")
df_cme.info()

✅ Столбец 'F_Volume_USD' успешно удален.
✅ Датасет фьючерсов успешно обновлен и сохранен как: cme_btc_futures_daily_final.csv

Первые 5 строк обновленного датасета (обратите внимание на округление):


,Date,F_Close,F_High,F_Low,F_Open,F_Volume_Contracts,F_Volume_USD_Mln
0,2017-12-18,19100.0,20650.0,18345.0,20650.0,1054,100.7
1,2017-12-19,18200.0,19725.0,17180.0,19135.0,559,50.9
2,2017-12-20,17040.0,18350.0,16435.0,17745.0,784,66.8
3,2017-12-21,15330.0,17270.0,15080.0,16400.0,879,67.4
4,2017-12-22,14135.0,15825.0,12265.0,15595.0,2374,167.8



Информация о типах данных и размере:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2163 entries, 0 to 2162
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                2163 non-null   object 
 1   F_Close             2163 non-null   float64
 2   F_High              2163 non-null   float64
 3   F_Low               2163 non-null   float64
 4   F_Open              2163 non-null   float64
 5   F_Volume_Contracts  2163 non-null   int64  
 6   F_Volume_USD_Mln    2163 non-null   float64
dtypes: float64(5), int64(1), object(1)
memory usage: 118.4+ KB
